# Topic 20d - Merge hazards and validation

Practice for section 5 of the [main lesson](topic20_data_wrangling.ipynb). This is the most important notebook in Topic 20.

A merge that goes wrong **usually does not raise an error**. It returns a DataFrame of a plausible size, full of plausible numbers, that is wrong. There are three ways this happens:

| Hazard | What happens | Symptom |
|---|---|---|
| **Orphan keys** | Rows vanish | Row count goes **down** after an inner join |
| **Duplicate keys** | Rows multiply | Row count goes **up** after any join |
| **Unclean keys** | Nothing matches | Row count is right, new columns are full of `NaN` |

All three are in the data you are about to use.

In [ ]:
from pathlib import Path

import pandas as pd


def find_data_dir():
    """Locate the lesson's data folder from any sensible working directory."""
    for candidate in [Path.cwd() / "data",
                      Path.cwd() / "data_wrangling" / "data",
                      Path.cwd().parent / "data"]:
        if (candidate / "orders.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find orders.csv - check your data folder.")


DATA_DIR = find_data_dir()
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 130)

orders = pd.read_csv(DATA_DIR / "orders.csv", parse_dates=["order_date"])
customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
regions = pd.read_csv(DATA_DIR / "regions.csv")
returns = pd.read_csv(DATA_DIR / "returns.csv")
orders["revenue_ghs"] = orders["quantity"] * orders["unit_price_ghs"]
print("loaded:", {"orders": len(orders), "customers": len(customers),
                  "products": len(products), "regions": len(regions),
                  "returns": len(returns)})

## 1. Check before you merge

Two cheap checks, run before any merge, catch two of the three hazards.

In [ ]:
def merge_report(left, right, key, name=""):
    """Report what a merge on `key` is about to do, before doing it."""
    left_keys = set(left[key].dropna())
    right_keys = set(right[key].dropna())
    orphans = left_keys - right_keys
    orphan_rows = left[key].isin(orphans).sum()
    right_duplicated = right[key].duplicated().sum()

    print(f"--- {name or key} ---")
    print(f"  left rows                {len(left):>6}")
    print(f"  right rows               {len(right):>6}")
    print(f"  keys missing from right  {len(orphans):>6}  "
          f"(affecting {orphan_rows} left rows)")
    print(f"  duplicate keys in right  {right_duplicated:>6}")

    if right_duplicated:
        print(f"  WARNING: a merge will ADD rows. Expected {len(left)}, "
              f"you will get more.")
    elif orphans:
        print(f"  NOTE: an inner join will DROP {orphan_rows} rows. "
              f"A left join keeps them with NaN.")
    else:
        print("  Clean: an m:1 left join will preserve the row count.")
    print()


merge_report(orders, customers, "customer_id", "orders -> customers")
merge_report(orders, products, "product_id", "orders -> products")
merge_report(orders, returns, "order_id", "orders -> returns")

That one function told you, before running a single merge, that `customers` will lose rows to an inner join and that `products` will **add** rows to any join.

## 2. Hazard 1 - orphan keys

Rows disappear, and nothing says so.

In [ ]:
true_revenue = orders["revenue_ghs"].sum()
inner = orders.merge(customers, on="customer_id", how="inner")
left = orders.merge(customers, on="customer_id", how="left")

print(f"{'':22}{'rows':>8}{'revenue GHS':>16}")
print(f"{'orders (truth)':22}{len(orders):>8}{true_revenue:>16,.2f}")
print(f"{'inner join':22}{len(inner):>8}{inner['revenue_ghs'].sum():>16,.2f}")
print(f"{'left join':22}{len(left):>8}{left['revenue_ghs'].sum():>16,.2f}")
print(f"\nThe inner join lost {len(orders) - len(inner)} orders worth "
      f"GHS {true_revenue - inner['revenue_ghs'].sum():,.2f}.")

### `indicator=True` names the problem rows

In [ ]:
checked = orders.merge(customers, on="customer_id", how="outer", indicator=True)
print(checked["_merge"].value_counts().to_string())
print("\nThe unknown customer ids:")
print(sorted(checked.loc[checked["_merge"] == "left_only", "customer_id"].unique()))

## 3. Hazard 2 - duplicate keys and the row explosion

The dangerous one, because the output looks entirely normal.

In [ ]:
exploded = orders.merge(products, on="product_id", how="left")
exploded["revenue_ghs"] = exploded["quantity"] * exploded["unit_price_ghs"]

print(f"rows before: {len(orders)}")
print(f"rows after:  {len(exploded)}   (+{len(exploded) - len(orders)} invented)")
print(f"\nrevenue before: GHS {true_revenue:,.2f}")
print(f"revenue after:  GHS {exploded['revenue_ghs'].sum():,.2f}")
print(f"overstated by:  GHS {exploded['revenue_ghs'].sum() - true_revenue:,.2f} "
      f"({exploded['revenue_ghs'].sum() / true_revenue - 1:.2%})")

print("\nThe culprit - one product_id, two rows:")
print(products[products["product_id"].duplicated(keep=False)].to_string(index=False))

print("\nAn order for that product now appears twice:")
print(exploded[exploded["product_id"] == "P011"]
      [["order_id", "product_id", "product_name", "supplier", "revenue_ghs"]]
      .head(4).to_string(index=False))

A 1.75% rise in row count and a 5% rise in revenue do not look like a bug. They look like a good year. Every row is a real product at a real price, there are no missing values, and nothing raised an error.

### `validate=` makes the assumption enforceable

| `validate=` | Means | Use for |
|---|---|---|
| `"m:1"` | Many left rows to one right row | **Joining a lookup onto a fact table - use constantly** |
| `"1:1"` | One to one | Two tables that should be the same list |
| `"1:m"` | One left to many right | Expanding a parent into children |
| `"m:m"` | Many to many | Almost always a mistake |

In [ ]:
try:
    orders.merge(products, on="product_id", how="left", validate="m:1")
    print("Merge succeeded.")
except pd.errors.MergeError as error:
    print("MergeError - the merge refused to run:")
    print(" ", str(error).split("\n")[0])
    print("\nThis is the outcome you want: an exception you cannot ignore,")
    print("instead of a DataFrame you will not question.")

### Your turn - fix it

1. Remove the duplicate key from `products` with `drop_duplicates(subset=..., keep="first")`.
2. Merge again with `validate="m:1"`.
3. Confirm the row count is back to 1,200 and the revenue total is correct.
4. Then answer in markdown: `keep="first"` chose one of the two supplier rows. On what basis? What would you do in a real project?

In [ ]:
# YOUR CODE HERE

**Your answer to question 4:**

*(Write your answer here.)*

## 4. Hazard 3 - an unclean key

Both tables have the column. Both have the values. The join still fails.

In [ ]:
naive = customers.merge(regions, on="region", how="left")
print(f"customers with no zone: {naive['zone'].isna().sum()} of {len(customers)}")

failed = sorted(set(customers["region"]) - set(regions["region"]))
print(f"\n{len(failed)} spellings failed to match, for 16 real regions.")
print("repr() shows the invisible characters:\n")
for row in range(0, min(len(failed), 12), 3):
    print("   " + "".join(f"{v!r:<22}" for v in failed[row:row + 3]))

In [ ]:
# The fix belongs BEFORE the join, and it is Topic 17 work.
customers_clean = customers.copy()
customers_clean["region"] = (customers_clean["region"]
                             .str.strip()
                             .str.replace(r"\s+", " ", regex=True)
                             .str.title())

fixed = customers_clean.merge(regions, on="region", how="left", validate="m:1")
print(f"unmatched before: {naive['zone'].isna().sum()}")
print(f"unmatched after:  {fixed['zone'].isna().sum()}")

## 5. The checklist

Run these every time:

1. `set(left[key]) - set(right[key])` - finds orphans.
2. `right[key].duplicated().any()` - finds explosions.
3. `validate="m:1"` on the merge - enforces your assumption.
4. Compare `len()` before and after - for an `m:1` left join it must not change.

### Final practice

Write a function `safe_merge(left, right, key, how="left", validate="m:1")` that:

- prints the row count before the merge;
- performs the merge with the given `how` and `validate`;
- prints the row count after, and whether it changed;
- raises a `ValueError` with a clear message if a left join changed the row count;
- returns the merged DataFrame.

Then use it to merge `customers` and `products_clean` onto `orders`, and show that it raises on the un-deduplicated `products`.

In [ ]:
# YOUR CODE HERE
# def safe_merge(left, right, key, how="left", validate="m:1"):
#     ...

---
## Where this fits

- [Back to the main lesson](topic20_data_wrangling.ipynb)
- Previous: [20c - Merging and joins](topic20c_merging_and_joins.ipynb)
- Next: [20e - Reshaping with melt and pivot](topic20e_reshaping_melt_and_pivot.ipynb)